<a href="https://colab.research.google.com/github/n2pakbong/boundary-adapted-pinns/blob/main/colab/periodic_committor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Periodic two-well committor

This notebook estimates the forward committor on a two-dimensional torus. Read the problem in Section 1, then edit the labeled problem, model, sampling, and training cells in Sections 2–5. Run top to bottom: Section 6 creates a fresh run, saves its seeds and numerical data, and trains the selected methods; Section 7 evaluates the saved checkpoints and creates the complete run archive. Drive is optional, and the notebook passes its output folder between cells.


## 1. Problem

Consider overdamped Langevin dynamics on the flat torus $\mathbb T^2=[0,1)^2$:

$$
dX_t=-\nabla V(X_t)\,dt+\sqrt{2\varepsilon}\,dW_t \pmod 1,
\qquad
V(x_1,x_2)=a\left[1-\cos\!\left(4\pi(x_1-\tfrac14)\right)\right]
+b\left[1-\cos(2\pi x_2)\right].
$$

For the default settings, $A$ and $B$ are disjoint periodic disks, centered at $(1/4,0)$ and $(3/4,0)$. The **forward committor** is

$$
q(x)=\mathbb P_x(\tau_B<\tau_A),
$$

the probability that a trajectory started at $x$ reaches $B$ before $A$. It solves

$$
\mathcal L q
=-\nabla V\cdot\nabla q+\varepsilon\Delta q=0
\quad\text{in }D=\mathbb T^2\setminus(A\cup B),
$$

with $q=0$ on $\partial A$ and $q=1$ on $\partial B$. Thus this notebook studies a hitting probability, not a mean escape time.

The starting values are $a=1$, $b=0.6$, $\varepsilon=0.4$, and periodic chordal radius $r=0.5$. The finite-difference solution is a numerical target for error measurement, not exact ground truth.

This notebook supports one problem family: a flat unit torus with two equal-radius periodic disks and the cosine potential shown above. You can change the potential amplitudes, diffusion strength, disk centers, and common radius below. The torus, disk shape, potential formula, and cosine phase centers stay fixed. Moving a disk does not move a potential well. The runner rejects disks that overlap or touch. A different domain, disk geometry, or potential formula requires code changes to the problem implementation and its samplers.

### Five methods compared

| Method | Trial function | Trained loss components | Boundary treatment |
|---|---|---|---|
| Standard PINN (`simple_pinn`) | $q_\theta=N_\theta$ | PDE, Monte Carlo data, ordinary boundary residual | Samples both boundaries and penalizes the Dirichlet residual. |
| Boundary-adapted | $g+\rho N_\theta$ | PDE, Monte Carlo data | Exact values from the ansatz; no boundary-residual penalty. |
| Over-enforced | $g+\rho^{3/2}N_\theta$ | PDE, Monte Carlo data | Exact values from the ansatz; no boundary-residual penalty. |
| Under-enforced | $g+\rho^{1/2}N_\theta$ | PDE, Monte Carlo data | Exact values from the ansatz; no boundary-residual penalty. |
| Variational/Nitsche | $q_\theta=N_\theta$ | Interior-plus-flux loss + Nitsche boundary loss + Monte Carlo data | Weakly imposes the boundary values using Nitsche's method. |

The code names appear in parentheses. All methods use the same PDE points and Monte Carlo labels; Section 3 shows the trial functions and Section 5 defines the loss terms.

## 2. Choose the run and set problem parameters

`DEVICE_NAME = "auto"` uses a GPU when one is available and otherwise uses the CPU. Set it to `"cpu"` or `"cuda"` to choose directly.

`SAVE_TO_DRIVE = False` keeps results in the Colab runtime's `/content/periodic_committor_runs` folder. Set it to `True` only if you want persistent Google Drive output; the notebook mounts Drive only when the preparation or a later run-selection cell needs it.

Each preparation creates a new folder and draws fresh training and Monte Carlo seeds. Its saved configuration, reference, Monte Carlo labels, training histories, checkpoints, predictions, metrics, and plots stay together. The notebook passes the folder path between cells automatically.

In [ ]:
# Choose the hardware and output location.
DEVICE_NAME = "auto"     # "auto", "cpu", or "cuda".
DTYPE_NAME = "float64"   # "float32" or "float64".

SAVE_TO_DRIVE = False  # True mounts Drive only when a run folder is needed.
LOCAL_RESULTS_DIR = "/content/periodic_committor_runs"
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/boundary-adapted-pinns-runs/periodic_committor"
DOWNLOAD_RUN_ARCHIVE = True  # Create the ZIP; download it automatically only with local Colab storage.

### Workflow

| Stage | Work | Compute |
|---|---|---|
| Setup | Clone the public repository and import its source code | Network once; no experiment computation |
| Preparation | Finite-difference reference and shared Monte Carlo labels | CPU |
| Training | Selected networks, training histories, and checkpoints | The selected device |
| Inspect/evaluate | Saved metrics, arrays, and figures | CPU |

Keep the same Colab runtime attached through preparation and training. If the runtime resets, rerun Setup and the settings cells; later cells find the saved run automatically. The problem, sampling, model, losses, optimizer, schedule, training budget, and reporting controls are editable below.

### Problem parameters

The domain stays the flat unit torus: each coordinate wraps around at 0 and 1. For readability, enter each center coordinate in the range from 0 up to (but not including) 1; for example, 0.25 is one quarter of the way around the torus. The coordinates wrap periodically. Set A is the source, where the committor is 0. Set B is the target, where it is 1. Both use the same periodic chordal radius. If the two closed disks overlap or touch, the run stops with an error. The radius must lie strictly between 0 and 1.

`POTENTIAL_A` and `POTENTIAL_B` are the amplitudes in the displayed cosine formula; `DIFFUSION_EPSILON` is the noise strength and must be positive. Moving a disk center does not move a cosine well. This interface cannot change the dimension, domain, disk shape, or potential formula.


In [ ]:
# Edit these ordinary Python values to change the example problem.
POTENTIAL_A = 1.0
POTENTIAL_B = 0.6
DIFFUSION_EPSILON = 0.4
SOURCE_CENTER = (0.25, 0.0)
TARGET_CENTER = (0.75, 0.0)
COMMON_CHORDAL_RADIUS = 0.5


### Choose methods

Edit the Python list in the next cell to choose methods. The same selected methods are prepared and trained; no separate code-check run is performed.

In [ ]:
METHODS_TO_RUN = [
    "simple_pinn",
    "boundary_adapted",
    "over_enforced",
    "under_enforced",
    "variational_nitsche",
]


### Load the repository code

This cell clones the public repository's default branch into Colab storage and adds its `src` folder to the Python path. There is no branch choice. It does not connect to Drive or start an experiment.

In [ ]:
from pathlib import Path
import subprocess
import sys

try:
    from google.colab import drive
except ImportError:
    drive = None

REPO = Path("/content/boundary-adapted-pinns")
REPO_URL = "https://github.com/n2pakbong/boundary-adapted-pinns.git"

if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
elif not (REPO / ".git").is_dir():
    raise RuntimeError(f"{REPO} exists but is not a Git clone.")

source_path = REPO / "src"
if not (source_path / "boundary_pinn" / "__init__.py").is_file():
    raise FileNotFoundError(f"Package source not found in {source_path}.")
if str(source_path) not in sys.path:
    sys.path.insert(0, str(source_path))

def run_output_parent():
    if SAVE_TO_DRIVE:
        if drive is None:
            raise RuntimeError("Drive saving is available in Colab; set SAVE_TO_DRIVE = False outside Colab.")
        drive_root = Path("/content/drive/MyDrive")
        if not drive_root.is_dir():
            drive.mount("/content/drive", force_remount=False)
        if not drive_root.is_dir():
            raise RuntimeError("Google Drive did not mount. Reconnect Colab or choose local output.")
        return Path(DRIVE_RESULTS_DIR).expanduser()
    return Path(LOCAL_RESULTS_DIR).expanduser()


def resolve_run_root(required_file, purpose, unfinished_only=False):
    current_root = globals().get("RUN_ROOT")
    if current_root:
        current_path = Path(current_root)
        usable = current_path.is_dir() and (current_path / required_file).is_file()
        if unfinished_only:
            usable = usable and not (current_path / "summary.json").is_file()
        if usable:
            return current_path

    output_parent = run_output_parent()
    candidates = [
        path for path in output_parent.glob("periodic_committor_*")
        if path.is_dir() and (path / required_file).is_file()
    ]
    if unfinished_only:
        candidates = [path for path in candidates if not (path / "summary.json").is_file()]
    candidates.sort(key=lambda path: path.stat().st_mtime, reverse=True)
    if not candidates:
        raise FileNotFoundError(f"No saved run for {purpose} was found in {output_parent}.")
    if len(candidates) == 1:
        selected = candidates[0]
    else:
        print(f"Choose the run to {purpose}:")
        for number, path in enumerate(candidates, start=1):
            print(f"  {number}. {path.name}")
        choice = int(input(f"Run number [1-{len(candidates)}]: "))
        if not 1 <= choice <= len(candidates):
            raise ValueError("Choose one of the listed run numbers.")
        selected = candidates[choice - 1]
    globals()["RUN_ROOT"] = str(selected)
    print("Using saved run:", selected)
    return selected

sys.path.insert(0, str(source_path))
print("Repository code:", source_path)


## 3. Define the network and PDE

The torus is represented by sine and cosine coordinates, so the network receives four features. Edit `PeriodicCommittorNetwork` in the next cell using ordinary PyTorch code: change its layers, activation, initialization, or replace it with another `nn.Module`. Keep the periodic input features and return one scalar per point.

In [ ]:
import torch
from torch import nn

if DEVICE_NAME == "auto":
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
else:
    DEVICE = DEVICE_NAME
if DEVICE.startswith("cuda") and not torch.cuda.is_available():
    raise RuntimeError("DEVICE_NAME requests CUDA, but no CUDA device is available.")
DTYPE = DTYPE_NAME
if DTYPE not in {"float32", "float64"}:
    raise ValueError("DTYPE_NAME must be 'float32' or 'float64'.")



class PeriodicCommittorNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(4, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 1),
        )
        # PyTorch initializes each nn.Linear when it is created. To change it,
        # add nn.init_* calls here, for example:
        # for layer in self.layers:
        #     if isinstance(layer, nn.Linear):
        #         nn.init.xavier_normal_(layer.weight)
        #         nn.init.zeros_(layer.bias)


    def forward(self, x):
        phase = 2.0 * torch.pi * x
        features = torch.cat((torch.sin(phase), torch.cos(phase)), dim=-1)
        return self.layers(features).squeeze(-1)


print(PeriodicCommittorNetwork())

### Boundary conditions and trial functions

For each boundary set $j\in\{A,B\}$, the code builds a smooth level-set function $\omega_j$ and a normalized distance-like function $d_j$:

$$
\omega_j(x)=4\sum_{k=1}^2\sin^2\!\bigl(\pi(x_k-c_{jk})\bigr)-r^2,
\qquad
d_j(x)=\frac{\omega_j(x)}
{\sqrt{\omega_j(x)^2+\lvert\nabla\omega_j(x)\rvert^2}}.
$$

Here $c_j$ is the center of set $j$. The boundary is where $\omega_j=0$; in the exterior domain, both $d_A$ and $d_B$ are positive. Define the smooth lift $g$ and the two-boundary factor $\rho$ by

$$
g(x)=\frac{d_A(x)}{d_A(x)+d_B(x)},
\qquad
\rho(x)=\frac{d_A(x)d_B(x)}
{\sqrt{d_A(x)^2+d_B(x)^2}}.
$$

On $\omega_j=0$, where $\nabla\omega_j\ne0$, differentiation gives $\nabla d_j=\nabla\omega_j/|\nabla\omega_j|$, hence $|\nabla d_j|=1$. This is the first-order normalization: near its own boundary, $d_j$ changes at unit rate in the normal direction.

To see why the two-set factor is also normalized, write $F(a,b)=ab/\sqrt{a^2+b^2}$, so $\rho=F(d_A,d_B)$. On $\partial A$, the sets are disjoint and $d_B>0$. Since $\partial_aF(0,d_B)=1$ and $\partial_bF(0,d_B)=0$, we get $\nabla\rho=\nabla d_A$ and $|\nabla\rho|=1$. The same calculation gives $\nabla\rho=\nabla d_B$ and unit gradient on $\partial B$. The outward normal of the exterior domain points into a removed set, so $\partial_{n_D}\rho=-1$ on either boundary.

A plain product $d_A d_B$ also vanishes on both boundaries, but its normal derivative is scaled by the distance to the other set. The normalized $\rho$ removes that scale. Let $s$ be distance measured into $D$ from either boundary. Since $\partial_{n_D}\rho=-1$, $\rho=s+O(s^2)$ there. Thus $\rho^\alpha N_\theta$ behaves like $s^\alpha N_\theta$: for $\alpha=1$, its normal derivative at the boundary is finite and equals $-N_\theta$; for $\alpha>1$, its first normal derivative vanishes; for $\alpha<1$, it can be singular when $N_\theta$ is nonzero on the boundary. Thus $g=0$ on $\partial A$, $g=1$ on $\partial B$, and $\rho=0$ on both. The three exact-boundary trial functions use

$$
q_\theta(x)=g(x)+\rho(x)^\alpha N_\theta(x).
$$

Boundary-adapted uses $\alpha=1$, over-enforced uses $\alpha=3/2$, and under-enforced uses $\alpha=1/2$. Each satisfies both Dirichlet values by construction. The network output is periodic because the network receives sine and cosine coordinates. For background on the standard PINN residual and exact distance-function boundary enforcement, see [Raissi et al. (2019)](https://doi.org/10.1016/j.jcp.2018.10.045) and [Sukumar and Srivastava (2022)](https://doi.org/10.1016/j.cma.2021.114333); the exponent variants listed here are the specific cases in this experiment.

## 4. Prepare samples and reference data

This function proposes points uniformly from `[0, 1)²` and rejects points inside A or B, so accepted points are uniform by area on the punctured torus D. It supplies both PDE collocation points and Monte Carlo anchors. Edit the function to use another distribution; every point must remain in D. A nonuniform law changes the empirical averages unless you add importance weights. Boundary points use a separate sampler: uniform radial angles with arclength weights in boundary losses and Nitsche integrals.

Monte Carlo anchor and path settings follow the collocation and optimizer settings in Section 6.

In [ ]:
def sample_training_points(problem, n, *, device, dtype, generator=None):
    # Uniform rejection sampling on D = [0, 1)^2 \\ (A union B).
    accepted = []
    remaining = n
    for _ in range(100):
        proposal = torch.rand(
            max(2 * remaining, 32), 2,
            device=device, dtype=dtype, generator=generator,
        )
        inside_D = problem.domain_mask(proposal)
        take = proposal[inside_D][:remaining]
        if len(take):
            accepted.append(take)
            remaining -= len(take)
        if remaining == 0:
            return torch.cat(accepted, dim=0)
    raise RuntimeError("Could not sample enough points in D; check the geometry.")

### Monte Carlo labels

The committor also has a stochastic representation. Starting at anchor $z_i$, independent Euler–Maruyama trajectories approximate the Langevin dynamics:

$$
X_{n+1}
=\left[X_n-\nabla V(X_n)\Delta t
+\sqrt{2\varepsilon\Delta t}\,\xi_n\right]\pmod 1,
\qquad \xi_n\sim\mathcal N(0,I_2).
$$

This carries over the pointwise solution-value regularizer used in the mean-escape-time notebooks: here the target value is the committor probability, and it complements rather than replaces the PDE residual. Each trajectory stops on first entering $A$ or $B$, or when it reaches the configured step cap. The label is the fraction of resolved trajectories that hit $B$ first:

$$
\widehat q_i
=\frac{\#\{\text{resolved trajectories from }z_i\text{ that hit }B\text{ first}\}}
{\#\{\text{resolved trajectories from }z_i\}}.
$$

Capped, unresolved paths are **not** counted as hits of $A$. Training stops with an error if an anchor's resolved fraction is below the configured minimum. The reported binomial standard error is conditional on the resolved paths; if paths remain unresolved, the guide explains the possible censoring bias and time-step error.

**What you will see while it runs:** reports show the step cap, paths resolved to (A) and (B), paths still active, anchors meeting the configured resolution threshold, the minimum and mean resolved fractions, and elapsed time. They appear at the configured fraction of the step cap or after the configured time interval, whichever comes first. The final resolution summary appears before training. After generation, the next cell saves and displays a plot of anchor locations colored by their estimated committor values, plus each label with its estimated Monte Carlo standard error. All methods share the same anchors and labels in `monte_carlo_labels.npz` and `monte_carlo_labels.json`. Neural-network approaches to committor computation are discussed, for example, by [Khoo, Lu, and Ying (2018)](https://arxiv.org/abs/1802.10275) and [Song, Cameron, and Yang (2023)](https://arxiv.org/abs/2306.12268).

## 5. Define losses and training settings

Set the total training point counts first. Then define the losses and the settings used by the optimizer.

### Total training and evaluation point counts

`COLLOCATION_POINTS` is the total number of interior points used in each method's PDE loss. The training cell samples this set once and reuses it at every update; it is not a per-update batch size. `BOUNDARY_POINTS` is the total number of points on each target boundary for methods that use boundary terms. Evaluation boundary points are separate from training points.

In [ ]:
COLLOCATION_POINTS = 1024
BOUNDARY_POINTS = 256
EVALUATION_BOUNDARY_POINTS = 512


Every method uses the same interior collocation points and the same Monte Carlo anchor labels within a comparison. Write the interior residual loss and shared solution-value data loss as

$$
\widehat J_{\rm pde}(\theta)
=\frac{1}{N_D}\sum_{i=1}^{N_D}
\left|\mathcal Lq_\theta(x_i)\right|^2,
\qquad
\widehat J_{\rm data}(\theta)
=\frac{1}{N_{\rm data}}\sum_{i=1}^{N_{\rm data}}
\left(q_\theta(z_i)-\widehat q_i\right)^2.
$$

Only the standard PINN has the ordinary boundary-residual component:

$$
\widehat J_{\rm bc}
=\frac{1}{N_A}\sum_i q_\theta(x_i^A)^2
+\frac{1}{N_B}\sum_i\bigl(q_\theta(x_i^B)-1\bigr)^2.
$$

The boundary-adapted, over-enforced, and under-enforced methods have **no extra boundary penalty**. The variational method's stabilization term belongs to the Nitsche weak formulation; it is not applied to an exact-boundary ansatz. The loss components and the Nitsche method are explained in the cells below.

### Adaptive weighting

The paper denotes the total adaptive objective by $\widehat J$ and its active empirical loss components by $\widehat J_i$. For each component, $\lambda_i>0$ is a trainable scale. Its objective (Eq. (33)) is

$$
\widehat J(\theta,\{\lambda_i\}_{i\in\mathcal I})
=\sum_{i\in\mathcal I}\frac{1}{2\lambda_i^2}\widehat J_i(\theta)
+\sum_{i\in\mathcal I}\log\lambda_i.
$$

See [Tepakbong et al. (2026), Eq. (33)](https://proceedings.mlr.press/v306/tepakbong26a.html). The active set $\mathcal I$ follows the method being trained:

- Standard PINN (simple_pinn): $\{\mathrm{pde},\mathrm{bc},\mathrm{data}\}$. Here $\widehat J_{\rm bc}$ is the sampled Dirichlet residual.
- Boundary-adapted, over-enforced, and under-enforced: $\{\mathrm{pde},\mathrm{data}\}$. Their boundary values are built into the ansatz, so there is no boundary-residual term.
- Variational/Nitsche: $\{\mathrm{pde},\mathrm{bc},\mathrm{data}\}$. Here $\widehat J_{\rm pde}$ is the squared interior-plus-flux residual, and $\widehat J_{\rm bc}$ is the squared Nitsche boundary contribution (symmetric consistency plus stabilization). This is not the standard PINN's pointwise boundary residual.

For this committor, $\widehat J_{\rm pde}$ uses the residual $\mathcal Lq_\theta=0$, and $\widehat J_{\rm data}$ compares $q_\theta(z_i)$ with the simulated committor label $\widehat q_i$. The loss definitions and Nitsche split are explained in the cells below. In the training logs, the three optimized Nitsche components are pde_loss, boundary_loss, and data_loss. The separate Nitsche term mean squares and the full weak-residual mean square are diagnostics, not separately weighted losses.

**How the code's saved scale corresponds to $\lambda_i$.** The code stores each trainable scale in log form, $s_i=\log\sigma_i$, and multiplies each raw loss by $e^{-2s_i}$. The paper's parameter is $\lambda_i=e^{s_i}/\sqrt{2}$, so $e^{-2s_i}=1/(2\lambda_i^2)$. Thus each weighted component has exactly the paper's coefficient. The code's total objective differs from the paper's displayed objective only by the constant $|\mathcal I|\log(2)/2$, which has zero gradient and does not change training. The raw loss components and effective weights are logged separately. The trainable-scale construction follows [Kendall, Gal, and Cipolla (2018)](https://openaccess.thecvf.com/content_cvpr_2018/html/Kendall_Multi-Task_Learning_Using_CVPR_2018_paper.html).

The notebook's `training.log_interval_seconds` setting controls the wall-clock interval between live console reports. The first and final epochs always print; each report includes the epoch, elapsed time, ETA, objective, and active raw loss components. Regardless of that interval, every epoch is written to `metrics.jsonl` with raw components, weighted components, effective weights, learning rate, and timing. Thus `losses.png` shows the full per-epoch history. `training.checkpoint_every` separately controls restartable checkpoints. Final raw loss components, the best epoch, and the objective are saved in `final_training_metrics.json` and `summary.json`.

Each trainable log scale starts at `training.adaptive_loss_log_scale_init` (zero in the pilot), then is optimized jointly with the network using the configured optimizer and learning-rate schedule. A zero starting value makes every raw loss coefficient initially one in the code objective.


In [ ]:
# Initial log scale for the trainable loss weights in Adaptive weighting.
ADAPTIVE_LOSS_LOG_SCALE_INIT = 0.0

### Variational/Nitsche method

Because the drift is a gradient, set $m(x)=e^{-V(x)/\varepsilon}$. The strong committor equation is equivalent to

$$
\mathcal Lq=0
\quad\Longleftrightarrow\quad
\nabla\cdot\left(\varepsilon m\nabla q\right)=0.
$$

For periodic Gaussian test functions $\phi_j$ centered at $c_j$, let $n_D$ be the outward normal of the exterior domain $D$, pointing into the removed sets:

$$
\phi_j(x)=\exp\!\left[
-\frac{4\sum_{k=1}^2\sin^2\!\bigl(\pi(x_k-c_{jk})\bigr)}{2\ell^2}
\right].
$$

The interior-plus-flux contribution is

$$
A_j=
\varepsilon\int_D m\nabla q_\theta\cdot\nabla\phi_j\,dx
-\varepsilon\int_{\partial D}m\,\partial_{n_D}q_\theta\,\phi_j\,ds.
$$

The symmetric consistency term and Nitsche stabilization moment are

$$
C_j=-\varepsilon\int_{\partial D}m(q_\theta-g)\,\partial_{n_D}\phi_j\,ds,
\qquad
P_j=\frac{\gamma\varepsilon}{h}
\int_{\partial D}m(q_\theta-g)\phi_j\,ds,
$$

where $g=0$ on $\partial A$ and $g=1$ on $\partial B$. Together they form the Nitsche boundary contribution $B_j=C_j+P_j$. The full weak residual is $R_j=A_j+B_j$.

The paper's VPINN loss split is

$$
\widehat J_{\rm pde}=\frac{1}{N_\phi}\sum_{j=1}^{N_\phi}A_j^2,
\qquad
\widehat J_{\rm bc}=\frac{1}{N_\phi}\sum_{j=1}^{N_\phi}B_j^2.
$$

These two losses are optimized with the common committor data term $\widehat J_{\rm data}$. In the training log they are named pde_loss, boundary_loss, and data_loss. Nitsche stabilization belongs inside the boundary component; it is not an extra loss on an exact-boundary ansatz or the standard PINN's pointwise boundary residual. The separate volume, flux, consistency, stabilization, and full-residual mean squares are diagnostics, not additional adaptive losses. This follows [Liao and Ming (2021)](https://doi.org/10.4208/cicp.OA-2020-0219) and [Berrone et al. (2023)](https://doi.org/10.1016/j.heliyon.2023.e18820).

**Check the equations in code:** the PDE and boundary geometry are in src/boundary_pinn/problems/committor/periodic_two_well.py; Monte Carlo and weak-form integrals are in paper_methods.py; ansatzes, active loss sets, adaptive scales, and epoch logs are in training.py; orchestration and saved error tables are in comparison.py.

### Nitsche integration settings

These values are used only when `variational_nitsche` is selected. `NITSCHE_VOLUME_POINTS` sets the number of interior quadrature points, `NITSCHE_TEST_FUNCTIONS` sets the number of Gaussian test functions, and `TEST_FUNCTION_LENGTH` sets their width. The saved boundary points are shared with the other methods. `STABILIZATION_GAMMA` and `BOUNDARY_SCALE_H` set the Nitsche boundary stabilization.

In [ ]:
# Used only when "variational_nitsche" is in METHODS_TO_RUN.
NITSCHE_VOLUME_POINTS = 1024
NITSCHE_TEST_FUNCTIONS = 32
TEST_FUNCTION_LENGTH = 0.12
STABILIZATION_GAMMA = 10.0
BOUNDARY_SCALE_H = 0.03

### Number of updates and optimizer

`EPOCHS` sets how many updates each selected method receives. The next cell constructs Adam using ordinary PyTorch arguments. Edit the learning rate or betas here, or replace `torch.optim.Adam` with another optimizer that supports this loop's `.step()` call.

The next cell constructs Adam using ordinary PyTorch arguments. Edit the learning rate or betas here, or replace `torch.optim.Adam` with another optimizer that supports this loop's `.step()` call.

In [ ]:
EPOCHS = 6000  # Number of updates for each selected method.
LEARNING_RATE = 0.001
OPTIMIZER_BETAS = (0.9, 0.999)


def make_optimizer(parameters):
    return torch.optim.Adam(parameters, lr=LEARNING_RATE, betas=OPTIMIZER_BETAS)


### Learning-rate schedule

The scheduler takes one step after every optimizer update. Edit `SCHEDULE_GAMMA` to change the exponential decay, or replace the constructor with another PyTorch scheduler whose `step()` takes no arguments.

In [ ]:
SCHEDULE_GAMMA = 0.9999


def make_scheduler(optimizer):
    return torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=SCHEDULE_GAMMA)


### Progress and restart checkpoints

The training call in Section 6 reports the current epoch, elapsed time, estimated time remaining, objective, and active loss components. It writes epoch history and periodic checkpoints to the run folder. Change `LOG_INTERVAL_SECONDS` to control report frequency and `CHECKPOINT_EVERY` to control how often restart checkpoints are written.

In [ ]:
LOG_INTERVAL_SECONDS = 30
CHECKPOINT_EVERY = 100

### Optional custom training update

By default, the five-method comparison uses the training implementation in the linked comparison module, with the losses defined above. Leave `TRAINING_STEP = None` for that paper comparison. The next cell is a complete example of replacing one update with a PDE-residual-only update; edit it or write your own function with the same arguments. The actual run starts in Section 6 after its fixed points and Monte Carlo labels have been prepared.

In [ ]:
# Optional complete replacement for the built-in one-update-per-epoch logic.
def residual_only_training_step(
    *, model, problem, config, epoch, optimizer, scheduler, loss_scales,
    mc_data, device, dtype, interior_sampler,
):
    sampler = interior_sampler or sample_training_points
    points = sampler(
        problem, int(config.training["collocation_points"]),
        device=device, dtype=dtype,
    )
    objective = problem.residual(model, points).square().mean()
    optimizer.zero_grad(set_to_none=True)
    objective.backward()
    optimizer.step()
    scheduler.step()
    return {"objective": objective.detach(), "pde_loss": objective.detach()}

# Keep None for the built-in five-method objectives.
TRAINING_STEP = None
# To use the residual-only example instead, assign: TRAINING_STEP = residual_only_training_step


### Reference grid and evaluation batch

After training, the notebook compares the network's predictions with the finite-difference solution prepared earlier. It does this at 65,536 grid points (256 points in each coordinate).

The network processes 8,192 points at a time, so it takes 8 calls to cover the grid. The notebook then compares all predictions with the finite-difference values and calculates the errors.

`EVALUATION_BATCH_SIZE` sets the number of points in each call. It affects only this final comparison, not training or the network. Leave it at 8192 unless evaluation runs out of memory; then try 4096.

In [ ]:
REFERENCE_GRID_SIZES = [128, 256]
EVALUATION_BATCH_SIZE = 8192  # Points processed at a time for the final error calculation.


## 6. Save data and train

Choose the Monte Carlo settings immediately below, then run the preparation cell. It computes the finite-difference reference, simulates committor labels, and saves the shared point sets. Preparation does not train a network. Run the training cell afterward; it uses the prepared folder and settings. After a runtime reset, the notebook locates the saved run in the selected storage.

### Monte Carlo labels for the committor

The committor is $q(z)=\mathbb P_z(\tau_B<\tau_A)$: the probability that a path starting at $z$ reaches target set $B$ before source set $A$. Independent Euler–Maruyama paths estimate this probability at each anchor. The label is the fraction of resolved paths that hit $B$ first. A path reaching the step cap without hitting either set is unresolved; it is not counted as a hit, and preparation stops if too few paths resolve.

Anchors are separate from PDE collocation points. Their labels add a pointwise solution-value loss alongside the PDE residual. The editable anchor fraction sets the number of starting locations; paths per anchor, time step, and step cap control simulation cost and label quality. This noisy Feynman–Kac supervision is analyzed by [Tepakbong et al., “Taming the Loss Landscape of PINNs with Noisy Feynman–Kac Supervision” (ICML 2026)](https://proceedings.mlr.press/v306/tepakbong26a.html).

In [ ]:
# Fraction of PDE collocation points used as Monte Carlo starting locations.
ANCHOR_FRACTION = 0.05
# Point counts are integers: int takes the whole-point part of the product.
N_ANCHORS = int(ANCHOR_FRACTION * COLLOCATION_POINTS)
if N_ANCHORS < 1:
    raise ValueError("Increase ANCHOR_FRACTION so at least one anchor is used.")
# More independent paths per anchor reduce Monte Carlo noise and take longer.
PATHS_PER_ANCHOR = 256
TIME_STEP = 0.001
MAX_STEPS = 100000
MINIMUM_RESOLVED_FRACTION = 0.9
PROGRESS_INTERVAL_FRACTION = 0.01
PROGRESS_INTERVAL_SECONDS = 10

In [ ]:
# One saved configuration captures the editable settings above.
from pathlib import Path
import json
import secrets

SEED = None                 # Assigned once when this preparation creates its run folder.
MONTE_CARLO_SEED = None     # Assigned once beside the training seed.

# Save exact settings with this run; training and resume read the same values.
RUN_CONFIG = {
    "seed": SEED,
    "device": DEVICE,
    "dtype": DTYPE,
    "methods": METHODS_TO_RUN,
    "problem": {
        "a": POTENTIAL_A,
        "b": POTENTIAL_B,
        "epsilon": DIFFUSION_EPSILON,
        "source_center": list(SOURCE_CENTER),
        "target_center": list(TARGET_CENTER),
        "set_radius": COMMON_CHORDAL_RADIUS,
    },
    "training": {
        "epochs": EPOCHS,
        "collocation_points": COLLOCATION_POINTS,
        "boundary_points": BOUNDARY_POINTS,
        "learning_rate": LEARNING_RATE,
        "checkpoint_every": CHECKPOINT_EVERY,
        "log_interval_seconds": LOG_INTERVAL_SECONDS,
        "adaptive_loss_log_scale_init": ADAPTIVE_LOSS_LOG_SCALE_INIT,
        "optimizer": {"name": "adam", "betas": list(OPTIMIZER_BETAS)},
        "scheduler": {"name": "exponential", "gamma": SCHEDULE_GAMMA},
        "monte_carlo": {
            "n_anchors": N_ANCHORS,
            "anchor_fraction": ANCHOR_FRACTION,
            "paths_per_anchor": PATHS_PER_ANCHOR,
            "dt": TIME_STEP,
            "max_steps": MAX_STEPS,
            "seed": MONTE_CARLO_SEED,
            "minimum_resolved_fraction": MINIMUM_RESOLVED_FRACTION,
            "progress_interval_fraction": PROGRESS_INTERVAL_FRACTION,
            "progress_interval_seconds": PROGRESS_INTERVAL_SECONDS,
        },
        "nitsche": {
            "volume_points": NITSCHE_VOLUME_POINTS,
            "test_functions": NITSCHE_TEST_FUNCTIONS,
            "boundary_points": BOUNDARY_POINTS,
            "length_scale": TEST_FUNCTION_LENGTH,
            "gamma": STABILIZATION_GAMMA,
            "h": BOUNDARY_SCALE_H,
        },
    },
    "reference_grid_sizes": REFERENCE_GRID_SIZES,
    "evaluation_batch_size": EVALUATION_BATCH_SIZE,
    "evaluation_boundary_points": EVALUATION_BOUNDARY_POINTS,
}

RUN_CONFIG_PATH = Path("/content/periodic_committor_selected_config.json")
print("Methods:", ", ".join(METHODS_TO_RUN))
print("Device and dtype:", DEVICE, DTYPE)
print("Settings will be saved with the prepared run:", RUN_CONFIG_PATH)


### Prepare the reference, labels, and shared points

This cell creates the reference solution, simulates committor labels, samples fixed collocation and boundary points, and saves those arrays in a unique folder with fresh seeds. It does not update network weights. The training cell below reads this folder and starts the optimizer; resume uses saved arrays rather than resampling.

In [ ]:
# This cell prepares and saves data only; network training starts in the next cell.
# Build the reference and Monte Carlo labels on CPU.
from datetime import datetime, timezone
from pathlib import Path
import json
import secrets
import torch
from IPython.display import Image, display
from boundary_pinn.experiments.committor.periodic_two_well.comparison import run_periodic_comparison

torch.set_num_threads(2)

# One fresh seed pair is generated for this new run and then saved with it.
SEED = secrets.randbelow(2**31)
MONTE_CARLO_SEED = secrets.randbelow(2**31)
RUN_CONFIG["seed"] = SEED
RUN_CONFIG["training"]["monte_carlo"]["seed"] = MONTE_CARLO_SEED

run_name = (
    "periodic_committor_"
    + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    + "_"
    + secrets.token_hex(4)
)
output_parent = run_output_parent()
output_parent.mkdir(parents=True, exist_ok=True)
RUN_ROOT = str(output_parent / run_name)
Path(RUN_ROOT).mkdir(parents=False, exist_ok=False)

RUN_CONFIG_PATH.write_text(
    json.dumps(RUN_CONFIG, indent=2) + "\n",
    encoding="utf-8",
)
print(f"Training seed: {SEED}; Monte Carlo seed: {MONTE_CARLO_SEED}", flush=True)
print("Run folder; the following training cell uses this automatically:", RUN_ROOT, flush=True)

run_periodic_comparison(
    RUN_CONFIG_PATH,
    Path(RUN_ROOT),
    prepare_only=True,
    interior_sampler=sample_training_points,
)
anchor_plot = Path(RUN_ROOT) / "monte_carlo_anchors.png"
if anchor_plot.is_file():
    print("Monte Carlo anchor plot:", anchor_plot)
    display(Image(filename=str(anchor_plot)))
print("Reference and Monte Carlo preparation are complete. No network training has run.", flush=True)

# Save the exact point sets shared by all methods. Each is generated once per fresh run.
import numpy as np
from boundary_pinn.problems.committor.periodic_two_well import PeriodicTwoWellCommittor
from boundary_pinn.problems.committor.paper_methods import sample_boundary_geometry

problem = PeriodicTwoWellCommittor(**RUN_CONFIG["problem"])
point_dtype = torch.float64 if DTYPE == "float64" else torch.float32
point_generator = torch.Generator(device="cpu").manual_seed(SEED)
fixed_points = {
    "collocation_points": sample_training_points(
        problem, COLLOCATION_POINTS, device=torch.device("cpu"),
        dtype=point_dtype, generator=point_generator,
    ),
}
if "variational_nitsche" in METHODS_TO_RUN:
    fixed_points["nitsche_volume_points"] = sample_training_points(
        problem, NITSCHE_VOLUME_POINTS, device=torch.device("cpu"),
        dtype=point_dtype, generator=point_generator,
    )
    fixed_points["nitsche_test_centers"] = sample_training_points(
        problem, NITSCHE_TEST_FUNCTIONS, device=torch.device("cpu"),
        dtype=point_dtype, generator=point_generator,
    )
for which in ("source", "target"):
    fixed_points[f"{which}_boundary"] = sample_boundary_geometry(
        problem, which, BOUNDARY_POINTS, device=torch.device("cpu"),
        dtype=point_dtype, generator=point_generator,
    )
    fixed_points[f"{which}_evaluation_boundary"] = sample_boundary_geometry(
        problem, which, EVALUATION_BOUNDARY_POINTS, device=torch.device("cpu"),
        dtype=point_dtype, generator=point_generator,
    )
training_point_path = Path(RUN_ROOT) / "training_points.npz"
training_point_arrays = {}
for name, value in fixed_points.items():
    if isinstance(value, tuple):
        for suffix, tensor in zip(("points", "normals", "weights"), value):
            training_point_arrays[f"{name}_{suffix}"] = tensor.cpu().numpy()
    else:
        training_point_arrays[name] = value.cpu().numpy()
training_point_arrays["training_seed"] = np.asarray(SEED, dtype=np.uint64)
temporary_point_path = training_point_path.with_name(training_point_path.name + ".tmp")
try:
    with temporary_point_path.open("wb") as stream:
        np.savez_compressed(stream, **training_point_arrays)
    temporary_point_path.replace(training_point_path)
finally:
    if temporary_point_path.exists():
        temporary_point_path.unlink()

manifest_path = Path(RUN_ROOT) / "run_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
manifest["artifacts"] = sorted(set(manifest.get("artifacts", [])) | {"training_points.npz"})
manifest["updated_utc"] = datetime.now(timezone.utc).isoformat()
temporary_manifest = manifest_path.with_name(manifest_path.name + ".tmp")
temporary_manifest.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
temporary_manifest.replace(manifest_path)
print("Saved fixed shared training and boundary-quadrature points:", training_point_path)



### Start training

This is the cell that starts network training. It passes the saved run folder, fixed point sets, optimizer and schedule constructors, and optional custom update to the comparison runner. The per-epoch five-method loop is in [the periodic committor comparison code](https://github.com/n2pakbong/boundary-adapted-pinns/blob/main/boundary_pinn/experiments/committor/periodic_two_well/comparison.py). Checkpoints and epoch histories are saved in the run folder.

In [ ]:
# This is the call that starts training from the saved preparation.
# The per-epoch loop is in boundary_pinn/experiments/committor/periodic_two_well/comparison.py.
from pathlib import Path
import json
import torch
from boundary_pinn.experiments.committor.periodic_two_well.comparison import run_periodic_comparison

run_root = resolve_run_root(
    "cpu_preparation.json",
    purpose="train",
    unfinished_only=True,
)
RUN_ROOT = str(run_root)
checkpoint_names = ("last.pt", "best.pt", "final.pt")
if any(path.is_file() for name in checkpoint_names for path in Path(RUN_ROOT).glob(f"*/{name}")):
    raise RuntimeError("This run already has training checkpoints. Use the resume cell or prepare a new run.")

prep_path = Path(RUN_ROOT, "cpu_preparation.json")
prepared_config_path = Path(RUN_ROOT, "comparison_config.json")
if not prep_path.is_file() or not prepared_config_path.is_file():
    raise FileNotFoundError(f"Saved preparation files are missing from {RUN_ROOT}.")
prepared_config = json.loads(prepared_config_path.read_text(encoding="utf-8"))

# Reuse the seed pair created during preparation, including after a runtime reset.
RUN_CONFIG["seed"] = int(prepared_config["seed"])
RUN_CONFIG["training"]["monte_carlo"]["seed"] = int(
    prepared_config["training"]["monte_carlo"]["seed"]
)
SEED = RUN_CONFIG["seed"]
MONTE_CARLO_SEED = RUN_CONFIG["training"]["monte_carlo"]["seed"]
RUN_CONFIG_PATH.write_text(json.dumps(RUN_CONFIG, indent=2) + "\n", encoding="utf-8")

# Reload the fixed shared points saved with this run; do not resample on resume.
import numpy as np
with np.load(Path(RUN_ROOT) / "training_points.npz", allow_pickle=False) as saved_points:
    fixed_training_data = {
        "collocation_points": torch.as_tensor(saved_points["collocation_points"]),
    }
    if "nitsche_volume_points" in saved_points.files:
        fixed_training_data["nitsche_volume_points"] = torch.as_tensor(
            saved_points["nitsche_volume_points"]
        )
        fixed_training_data["nitsche_test_centers"] = torch.as_tensor(
            saved_points["nitsche_test_centers"]
        )
    for which in ("source", "target"):
        fixed_training_data[f"{which}_boundary"] = tuple(
            torch.as_tensor(saved_points[f"{which}_boundary_{suffix}"])
            for suffix in ("points", "normals", "weights")
        )
        fixed_training_data[f"{which}_evaluation_boundary"] = tuple(
            torch.as_tensor(saved_points[f"{which}_evaluation_boundary_{suffix}"])
            for suffix in ("points", "normals", "weights")
        )

training_device = str(RUN_CONFIG["device"])
if training_device.startswith("cuda") and not torch.cuda.is_available():
    raise RuntimeError("This run requests CUDA, but the current runtime has no GPU.")

print(f"Training on {training_device.upper()} and saving results in {RUN_ROOT}", flush=True)
run_periodic_comparison(
    RUN_CONFIG_PATH,
    Path(RUN_ROOT),
    train_prepared=True,
    model_factory=PeriodicCommittorNetwork,
    interior_sampler=sample_training_points,
    optimizer_factory=make_optimizer,
    scheduler_factory=make_scheduler,
    training_step=TRAINING_STEP,
    fixed_training_data=fixed_training_data,
)


## 7. Evaluate and report

Run evaluation after training. It reads the saved reference and checkpoints, reports the error metrics, and saves the reference, predictions, signed and absolute errors, and figures. Signed-error plots share a common symmetric color range, recorded with the arrays and summary. The notebook then creates a ZIP containing the complete run, including samples, Monte Carlo labels, checkpoints, histories, metrics, and figures. With Drive selected, the ZIP stays beside the run; with local Colab storage, `DOWNLOAD_RUN_ARCHIVE` controls automatic download.

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display
import pandas as pd

run_root = resolve_run_root("summary.json", purpose="inspect completed results")
RUN_ROOT = str(run_root)

with open(f'{RUN_ROOT}/summary.json') as handle:
    summary = json.load(handle)

method_summaries = summary.get('methods', {})
accuracy_fields = [
    'grid_l2_abs', 'grid_l2_rel',
    'grid_h1_abs', 'grid_h1_rel',
    'grid_h1_seminorm_abs', 'grid_h1_seminorm_rel',
    'grid_h2_abs', 'grid_h2_rel',
    'grid_h2_seminorm_abs', 'grid_h2_seminorm_rel',
]
if method_summaries and all(
    all(field in metrics for field in accuracy_fields)
    for metrics in method_summaries.values()
):
    display(pd.DataFrame(method_summaries).T[accuracy_fields])
else:
    print(
        'This run summary predates the full Sobolev error report. '
        'Run the evaluation section to calculate absolute and relative L2, H1, and H2 errors '
        'from its saved checkpoints on CPU; no GPU retraining is needed.'
    )

print('Final raw training losses (not solution-error metrics):')
training_rows = {}
for method, metrics in method_summaries.items():
    training = metrics.get('final_training_metrics', {})
    raw_components = training.get('raw_loss_components', {})
    if not raw_components:
        metrics_path = Path(RUN_ROOT) / method / 'metrics.jsonl'
        if metrics_path.is_file():
            lines = [line for line in metrics_path.read_text().splitlines() if line.strip()]
            if lines:
                last = json.loads(lines[-1])
                raw_components = {
                    key: value for key, value in last.items()
                    if key.endswith('_loss') or key.endswith('_mse')
                }
                training = {
                    'final_training_objective': last.get('objective', last.get('loss')),
                    'raw_loss_components': raw_components,
                }
    training_rows[method] = {
        'final objective': training.get('final_training_objective'),
        **raw_components,
    }
display(pd.DataFrame(training_rows).T)

custom_training_rows = {}
for method in method_summaries:
    method_dir = Path(RUN_ROOT) / method
    method_metadata_path = method_dir / 'metadata.json'
    method_metadata = json.loads(method_metadata_path.read_text()) if method_metadata_path.is_file() else {}
    training_hook = method_metadata.get('runtime_hooks', {}).get('training_step', 'repository training step')
    if training_hook != 'repository training step':
        final_metrics = method_summaries[method].get('final_training_metrics', {}).get('final_epoch_metrics', {})
        if not final_metrics:
            metrics_path = method_dir / 'metrics.jsonl'
            if metrics_path.is_file():
                rows = [json.loads(line) for line in metrics_path.read_text().splitlines() if line.strip()]
                if rows:
                    final_metrics = {
                        key: value for key, value in rows[-1].items()
                        if key not in {'epoch', 'loss', 'session_elapsed_seconds',
                                       'session_epochs_per_second', 'estimated_remaining_seconds'}
                    }
        custom_training_rows[method] = final_metrics
if custom_training_rows:
    print('Final values returned by the custom training-step callback:')
    display(pd.DataFrame(custom_training_rows).T)

print('Finite-difference grid comparison:')
print(json.dumps(summary.get('reference_convergence', {}), indent=2))
display(Image(filename=f'{RUN_ROOT}/losses.png'))
for method in method_summaries:
    figure = Path(RUN_ROOT) / f'committor_{method}.png'
    if figure.is_file():
        display(Image(filename=str(figure)))

import shutil
archive_base = Path(RUN_ROOT).parent / f"{Path(RUN_ROOT).name}_archive"
archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=RUN_ROOT)
print("Complete run archive:", archive_path)

if not SAVE_TO_DRIVE and DOWNLOAD_RUN_ARCHIVE:
    try:
        from google.colab import files
    except ImportError:
        print("Automatic download is available in Colab; the ZIP is saved at the path above.")
    else:
        files.download(archive_path)


### Re-evaluate saved checkpoints

This separate cell reloads each method's `best.pt` checkpoint and recomputes errors against the saved finite-difference reference. It uses CPU only, does not retrain, and does not overwrite the original `summary.json`. It displays the Sobolev errors and final raw training-loss components and writes `saved_model_evaluation.json` to the same run folder.

In the same runtime, it uses the current `RUN_ROOT`. After a runtime reset, it finds saved runs automatically; if there are several, choose one by its number. Run the model and initialization cells first. Checkpoints contain weights, not your custom Python class, so use the same model code that created them.

In [ ]:
# Re-evaluate saved best.pt checkpoints on CPU; this does not train.
from pathlib import Path
import json
from IPython.display import display
import pandas as pd
from boundary_pinn.experiments.committor.periodic_two_well.comparison import evaluate_saved_models

run_root = resolve_run_root("summary.json", purpose="re-evaluate saved checkpoints")
RUN_ROOT = str(run_root)

# Uses the PeriodicCommittorNetwork class defined above. Restore that class
# from the training notebook before loading checkpoints after changing it.
evaluate_saved_models(Path(RUN_ROOT), model_factory=PeriodicCommittorNetwork)
report_path = Path(RUN_ROOT) / 'saved_model_evaluation.json'
report = json.loads(report_path.read_text(encoding='utf-8'))
print('Training code revision:', report.get('training_git_revision') or 'not recorded for this older run')
print('FDM reference:', report['reference_file'])
display_fields = [
    'grid_l2_abs', 'grid_l2_rel', 'grid_h1_abs', 'grid_h1_rel',
    'grid_h1_seminorm_abs', 'grid_h1_seminorm_rel',
    'grid_h2_abs', 'grid_h2_rel', 'grid_h2_seminorm_abs', 'grid_h2_seminorm_rel',
    'source_boundary_rmse', 'target_boundary_rmse',
]
display(pd.DataFrame(report['methods']).T[display_fields])
print('Full saved report:', report_path)


### Resume an interrupted run

Run the next cell only if training stopped before the selected methods completed. It finds the prepared run automatically, restores its saved settings and seeds, and continues from the method checkpoints. If several unfinished runs are available, it asks you to choose one by number. Do not change the model, sampler, loss, optimizer, schedule, or training settings before resuming.

In [ ]:
# Resume a run that stopped before all selected methods finished.
from pathlib import Path
import json
import torch
from boundary_pinn.experiments.committor.periodic_two_well.comparison import run_periodic_comparison

run_root = resolve_run_root(
    "cpu_preparation.json",
    purpose="resume",
    unfinished_only=True,
)
RUN_ROOT = str(run_root)
config_path = Path(RUN_ROOT, "comparison_config.json")
if not config_path.is_file():
    raise FileNotFoundError(f"Saved settings are missing from {RUN_ROOT}.")
saved_config = json.loads(config_path.read_text(encoding="utf-8"))

current_config = json.loads(json.dumps(RUN_CONFIG))
current_config["seed"] = saved_config["seed"]
current_config["training"]["monte_carlo"]["seed"] = saved_config["training"]["monte_carlo"]["seed"]
if current_config != saved_config:
    raise RuntimeError("Settings differ from the interrupted run. Restore them before resuming.")

RUN_CONFIG = saved_config
SEED = int(RUN_CONFIG["seed"])
MONTE_CARLO_SEED = int(RUN_CONFIG["training"]["monte_carlo"]["seed"])
RUN_CONFIG_PATH.write_text(json.dumps(RUN_CONFIG, indent=2) + "\n", encoding="utf-8")

# Reload the fixed shared points saved with this run; do not resample on resume.
import numpy as np
with np.load(Path(RUN_ROOT) / "training_points.npz", allow_pickle=False) as saved_points:
    fixed_training_data = {
        "collocation_points": torch.as_tensor(saved_points["collocation_points"]),
    }
    if "nitsche_volume_points" in saved_points.files:
        fixed_training_data["nitsche_volume_points"] = torch.as_tensor(
            saved_points["nitsche_volume_points"]
        )
        fixed_training_data["nitsche_test_centers"] = torch.as_tensor(
            saved_points["nitsche_test_centers"]
        )
    for which in ("source", "target"):
        fixed_training_data[f"{which}_boundary"] = tuple(
            torch.as_tensor(saved_points[f"{which}_boundary_{suffix}"])
            for suffix in ("points", "normals", "weights")
        )
        fixed_training_data[f"{which}_evaluation_boundary"] = tuple(
            torch.as_tensor(saved_points[f"{which}_evaluation_boundary_{suffix}"])
            for suffix in ("points", "normals", "weights")
        )

training_device = str(RUN_CONFIG["device"])
if training_device.startswith("cuda") and not torch.cuda.is_available():
    raise RuntimeError("This saved run needs a GPU runtime. Choose a GPU before resuming.")

run_periodic_comparison(
    RUN_CONFIG_PATH,
    Path(RUN_ROOT),
    train_prepared=True,
    resume=True,
    model_factory=PeriodicCommittorNetwork,
    interior_sampler=sample_training_points,
    optimizer_factory=make_optimizer,
    scheduler_factory=make_scheduler,
    training_step=TRAINING_STEP,
    fixed_training_data=fixed_training_data,
)
print(f"Resumed training on {training_device.upper()} in {RUN_ROOT}.")


## 8. References
- Y. Liao and P. Ming, “Deep Nitsche Method: Deep Ritz Method with Essential Boundary Conditions,” *Communications in Computational Physics* 29(5) (2021), 1365–1384. [doi:10.4208/cicp.OA-2020-0219](https://doi.org/10.4208/cicp.OA-2020-0219).
- M. Raissi, P. Perdikaris, and G. E. Karniadakis, “Physics-informed neural networks,” *Journal of Computational Physics* 378 (2019), 686–707. [doi:10.1016/j.jcp.2018.10.045](https://doi.org/10.1016/j.jcp.2018.10.045).
- N. Sukumar and A. Srivastava, “Exact imposition of boundary conditions with distance functions in physics-informed deep neural networks,” *Computer Methods in Applied Mechanics and Engineering* 389 (2022), 114333. [doi:10.1016/j.cma.2021.114333](https://doi.org/10.1016/j.cma.2021.114333).
- S. Berrone, C. Canuto, M. Pintore, and N. Sukumar, “Enforcing Dirichlet boundary conditions in physics-informed neural networks and variational physics-informed neural networks,” *Heliyon* 9(8) (2023), e18820. [doi:10.1016/j.heliyon.2023.e18820](https://doi.org/10.1016/j.heliyon.2023.e18820).
- A. Kendall, Y. Gal, and R. Cipolla, “Multi-task learning using uncertainty to weigh losses for scene geometry and semantics,” *CVPR* (2018). [paper](https://openaccess.thecvf.com/content_cvpr_2018/html/Kendall_Multi-Task_Learning_Using_CVPR_2018_paper.html).
- Y. Khoo, J. Lu, and L. Ying, “Solving for high dimensional committor functions using artificial neural networks,” [arXiv:1802.10275](https://arxiv.org/abs/1802.10275) (2018).
- Z. Song, M. K. Cameron, and H. Yang, “A Finite Expression Method for Solving High-Dimensional Committor Problems,” [arXiv:2306.12268](https://arxiv.org/abs/2306.12268) (2023).
- N. Tepakbong, J. Fan, X. Zhou, and D.-X. Zhou, “Boundary-Adapted PINNs for Elliptic Dirichlet Problems,” [arXiv:2607.19167](https://arxiv.org/abs/2607.19167) (2026).

- N. Tepakbong, H. Hu, C. Liu, and X. Zhou, “Taming the Loss Landscape of PINNs with Noisy Feynman–Kac Supervision: Operator Preconditioning and Non-Asymptotic Error Bounds,” ICML 2026. [arXiv preprint](https://arxiv.org/abs/2607.19167).
- N. Tepakbong, H. Hu, C. Liu, and X. Zhou, [“Taming the Loss Landscape of PINNs with Noisy Feynman–Kac Supervision: Operator Preconditioning and Non-Asymptotic Error Bounds”](https://proceedings.mlr.press/v306/tepakbong26a.html), *Proceedings of the 43rd International Conference on Machine Learning*, PMLR 306, 120120–120175, 2026.
